In [17]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader 

In [18]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_classes = 10
batch_size = 64
learning_rate = 1e-3
num_epochs = 5

In [ ]:
# CIAR10 의 경우 32×32 픽셀, 3채널 (RGB) 컬러 이미지.
# VGG 는 ImageNet 기반이라 입력을 ImageNet 통계로 정규화하고 224x224 로 키움
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [20]:
train_dataset = torchvision.datasets.CIFAR10(
    root='dataset/', train=True, transform=transform, download=True
)
test_dataset = torchvision.datasets.CIFAR10(
    root='dataset/', train=False, transform=transform, download=False
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


Files already downloaded and verified


In [ ]:
# model = torchvision.models.vgg16(pretrained=True)
model = torchvision.models.vgg16(weights=torchvision.models.VGG16_Weights.DEFAULT)

# model features layer parameter freezing 
for param in model.features.parameters():
    param.requires_grad = False

print(model)

/Users/soonbeom/anaconda3/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/soonbeom/anaconda3/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [22]:
model.classifier[6] = nn.Linear(4096, num_classes)
model = model.to(device)

print(model)

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [23]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=learning_rate,
)

In [24]:
def train():
    model.train()
    for epoch in range(num_epochs):
        running_loss = 0.0
        for batch_idx, (data, targets) in enumerate(train_loader):
            data = data.to(device)  
            targets = targets.to(device)
            
            scores = model(data)
            loss = criterion(scores, targets)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            
            if batch_idx % 100 == 0:
                print(f'Epoch [{epoch+1} / {num_epochs}] '
                      f'Batch [{batch_idx} / {len(train_loader)}] '
                      f'Loss: {loss.item(): .4f}')     
        
        avg_loss = running_loss / len(train_loader)
        print(f'====> Epoch {epoch+1} avg loss: {avg_loss:.4f}') 

In [25]:
def check_accuracy(loader, model, split='test'):
    print(f'Checking accuracy on {split} data')
    num_correct = 0
    num_samples = 0
    model.eval()

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)

            scores = model(x)
            _, preds = scores.max(dim=1)
            num_correct += (preds == y).sum().item()
            num_samples += preds.size(0)

    acc = num_correct / num_samples * 100
    print(f'Got {num_correct}/{num_samples} with accuracy {acc:.2f}%')
    model.train()
    return acc


In [26]:
if __name__ == '__main__':
    train()
    check_accuracy(train_loader, model, split='train')
    check_accuracy(test_loader, model, split='test')

Epoch [1 / 5] Batch [0 / 782] Loss:  2.4491
Epoch [1 / 5] Batch [100 / 782] Loss:  1.0313


KeyboardInterrupt: 

## 핵심 포인트 몇 가지:

1. 왜 features 만 얼리고 classifier 는 학습시키는가

VGG 구조는 크게 두 부분이다.:

model.features — Conv 레이어들 (이미지에서 패턴 추출). ImageNet 에서 배운 범용 feature 가 유용하니까 재사용
model.classifier — Linear 레이어들 (추출된 feature 로 분류). ImageNet 1000 클래스용이니까 내 태스크 (10클래스) 에 맞게 재학습
requires_grad = False 로 하면 역전파 시 gradient 계산 안 되고 업데이트도 안됨. 연산량도 줄어서 학습이 훨씬 빨라짐.

2. 왜 마지막 Linear 만 바꾸는가

classifier[6] 이 Linear(4096, 1000) 인데, 1000 은 ImageNet 클래스 수이다. CIFAR10 은 10 클래스니까 Linear(4096, 10) 으로 교체. 앞의 두 Linear (4096→4096) 은 그대로 쓰되 학습은 되게 둠 (requires_grad 를 따로 False 로 설정 안 했으니까 기본 True).

3. 단계별 파인튜닝 전략

진짜로 성능 쥐어짤 때는:

1단계 (코드의 현재 상태): features 얼리고 classifier 만 학습. 몇 epoch 빨리 수렴
2단계: features 의 일부 를 풀어주고 낮은 lr (예: 1e-4 이하) 로 전체 미세 조정
## 2단계 예시
for param in model.features[-6:].parameters():  # 마지막 몇 레이어만 풀기
    param.requires_grad = True

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-5,   # 더 작게
)
4. 메모리 주의

VGG16 + 224×224 입력은 메모리 많이 먹는다. OOM 나면 batch_size 를 줄여야 한다. (32, 16까지 가능). GPU 메모리 8GB 미만이면 32 추천.

5. 속도

CIFAR10 은 train 50000 장 + 224×224 라 1 epoch 에 GPU 에서도 몇 분 걸린다..
